In [ ]:
import os
import time
import json
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import datetime
from torch.utils.tensorboard import SummaryWriter

if torch.cuda.is_available():
    print("GPU is available")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available, using CPU")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# データフォルダ
standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Standardized1"
label_data_folder = "/home/nishioka/GNN/DefectLabels_4x4_test1"

# 座標データのロード
x_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy")[:3654]
y_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy")[:3654]
z_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy")[:3654]

# エッジ情報を読み込む
edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long).to(device)

# データとラベルファイルを対応付け
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Standardized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectLabel_L")]

# 欠陥なしのファイルを追加
data_files.append("Standardized1_Defect4x4_ELNOD_L19B45.npy")
label_files.append("DefectLabel_L19B45.npy")

# データとラベルのペアを取得する関数
def extract_layer_block(file_name):
    if file_name.startswith("0"):
        return (0, 0)  # 欠陥なしデータの場合
    try:
        layer_block_str = file_name.split("_")[-1].replace(".npy", "")
        layer_str = layer_block_str.split("L")[1].split("B")[0]
        block_str = layer_block_str.split("B")[1]
        layer = int(layer_str)
        block = int(block_str)
        return (layer, block)
    except (ValueError, IndexError):
        print(f"無効なファイル名の形式: {file_name}")
        return None

# データとラベルのペアを作成
data_label_pairs = {}
for data_file in data_files:
    layer_block = extract_layer_block(data_file)
    if layer_block:
        data_label_pairs[layer_block] = {"data": data_file}

for label_file in label_files:
    layer_block = extract_layer_block(label_file)
    if layer_block and layer_block in data_label_pairs:
        data_label_pairs[layer_block]["label"] = label_file

# 有効なペアのみ取得
valid_pairs = [(v["data"], v["label"]) for k, v in data_label_pairs.items() if "label" in v]

# サンプルをトレーニングデータとして取得
train_pairs = [("Standardized1_Defect4x4_ELNOD_L19B45.npy", "DefectLabel_L19B45.npy")]

# 残りのデータを1:1でバリデーションとテストに分割
val_pairs, test_pairs = train_test_split(valid_pairs, test_size=0.5, random_state=42)

# データ準備の関数
def prepare_data(pairs):
    sampled_data = []
    sampled_labels = []
    
    for data_file, label_file in pairs:
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        # データとラベルを読み込む
        values = np.load(data_file_path)[:3654]
        label = np.load(label_file_path)[:3654]
        
        # 座標データと応力データを結合してノード特徴量を作成
        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        sampled_data.append(node_features)
        sampled_labels.append(label)
    
    # データを結合しテンソルに変換
    sampled_data = np.stack(sampled_data)
    sampled_labels = np.hstack(sampled_labels)

    x = torch.tensor(sampled_data.reshape(-1, 4), dtype=torch.float).to(device)
    y = torch.tensor(sampled_labels, dtype=torch.float).to(device)

    return x, y

# 各データセットを準備
train_x, train_y = prepare_data(train_pairs)
val_x, val_y = prepare_data(val_pairs)
test_x, test_y = prepare_data(test_pairs)

# Dataオブジェクトにまとめる
train_data = Data(x=train_x, edge_index=edge_index, y=train_y)
val_data = Data(x=val_x, edge_index=edge_index, y=val_y)
test_data = Data(x=test_x, edge_index=edge_index, y=test_y)

# GATモデルの定義 (5層)
class GATModel(torch.nn.Module):  # 変更
    def __init__(self, hidden_channels=128):
        super(GATModel, self).__init__()
        self.conv1 = GATConv(4, hidden_channels, heads=4)  
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels * 2, heads=4)  
        self.conv3 = GATConv(hidden_channels * 8, hidden_channels * 4, heads=4)  
        self.conv4 = GATConv(hidden_channels * 16, hidden_channels, heads=4)  
        self.conv5 = GATConv(hidden_channels * 4, hidden_channels)  # 新しい5層目
        self.fc_defect = torch.nn.Linear(hidden_channels, 1)  # 回帰タスク用に出力1次元

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = F.relu(self.conv4(x, edge_index))
        x = F.relu(self.conv5(x, edge_index))  # 新しい5層目
        x = self.fc_defect(x)  
        return torch.sigmoid(x)  

# ハイパーパラメータ設定
hidden_channels = 128
learning_rate = 0.001
batch_size = 32
epochs = 2000
weight_decay = 5e-4
patience = 500  # Early Stopping用

# モデル、損失関数、オプティマイザの定義
model = GATModel(hidden_channels=hidden_channels).to(device)  
model.apply(init_weights)  # Xavier初期化
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
loss_fn = torch.nn.MSELoss()

# TensorBoardの初期化
writer = SummaryWriter(log_dir=f'/home/nishioka/GNN/GNNlogs/{type(model).__name__}_{timestamp}')

# EarlyStopping用の変数
best_val_loss = float('inf')
counter = 0

# 学習時間の計測
start_time = time.time()

# データをサンプルごとに分割
train_data_list = []
val_data_list = []
test_data_list = []

# サンプル単位でデータをDataLoaderに渡す
num_train_samples = train_x.shape[0] // 3654
num_val_samples = val_x.shape[0] // 3654
num_test_samples = test_x.shape[0] // 3654

for i in range(num_train_samples):
    train_data_list.append(Data(x=train_x[i*3654:(i+1)*3654], edge_index=edge_index, y=train_y[i*3654:(i+1)*3654]))

for i in range(num_val_samples):
    val_data_list.append(Data(x=val_x[i*3654:(i+1)*3654], edge_index=edge_index, y=val_y[i*3654:(i+1)*3654]))

for i in range(num_test_samples):
    test_data_list.append(Data(x=test_x[i*3654:(i+1)*3654], edge_index=edge_index, y=test_y[i*3654:(i+1)*3654]))

# データローダーの作成
train_loader = DataLoader(train_data_list, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data_list, batch_size=32,
val_loader = DataLoader(val_data_list, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data_list, batch_size=32, shuffle=False)

# EarlyStopping クラス定義と学習ループ、評価などは、GCN の場合と同じ
class EarlyStopping:
    """Early stopping utility to stop training when validation loss doesn't improve."""
    def __init__(self, patience=patience, verbose=False, delta=0, path='/home/nishioka/GNN/{type(model).__name__}checkpoint1.pt', trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

# EarlyStoppingクラスのインスタンス作成
early_stopping = EarlyStopping(patience=patience, verbose=True, path=f'/home/nishioka/GNN/GNNmodel/{type(model).__name__}_best_{timestamp}.pth')

train_losses = []
val_losses = []
test_losses = []
start_time = time.time()

# 学習ループの開始
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch.x = batch.x.to(device)
        batch.y = batch.y.to(device)
        batch.edge_index = batch.edge_index.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch.y.view(-1, 1).float().to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_losses.append(total_loss / len(train_loader))

    # 検証ステップ
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch.x = batch.x.to(device)
            batch.y = batch.y.to(device)
            batch.edge_index = batch.edge_index.to(device)
            out = model(batch)
            loss = loss_fn(out, batch.y.view(-1, 1).float().to(device))
            val_loss += loss.item()

    val_losses.append(val_loss / len(val_loader))

    # EarlyStoppingに現在のバリデーション損失とモデルを渡す
    early_stopping(val_loss / len(val_loader), model)

    # EarlyStoppingがTrueなら訓練を終了
    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch}")
        break

    # 1, 10, 30, 50, 100, 150, 200, 250... のタイミングで結果を出力
    if epoch == 1 or epoch == 10 or epoch == 30 or (epoch % 50 == 0):
        elapsed_time = time.time() - start_time
        print(f'Epoch {epoch}, Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}, Time: {elapsed_time:.2f}s')

# テストデータでの評価
model.eval()
test_loss = 0
test_preds = []
test_targets = []
with torch.no_grad():
    for batch in test_loader:
        batch.x = batch.x.to(device)
        batch.y = batch.y.to(device)
        batch.edge_index = batch.edge_index.to(device)
        out = model(batch)
        loss = loss_fn(out, batch.y.view(-1, 1).float().to(device))
        test_loss += loss.item()
        test_preds.append(out.cpu().numpy())
        test_targets.append(batch.y.cpu().numpy())

test_loss /= len(test_loader)
test_preds = np.concatenate(test_preds)
test_targets = np.concatenate(test_targets)

# 評価指標
rmse = np.sqrt(mean_squared_error(test_targets, test_preds))
mae = mean_absolute_error(test_targets, test_preds)
r2 = r2_score(test_targets, test_preds)

print(f'Test Loss: {test_loss:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2 Score: {r2:.4f}')

# 学習時間の計測終了
elapsed_time = time.time() - start_time
print(f"Total training time: {elapsed_time:.2f} seconds")

# モデル、学習結果の保存
model_path = f'/home/nishioka/GNN/GNNmodel/{type(model).__name__}_final_{timestamp}.pth'
torch.save(model.state_dict(), model_path)
print(f"Model saved at {model_path}")

# 学習時間と損失の保存
with open(f'/home/nishioka/GNN/GNNmodelcsv/training_time_{timestamp}.txt', 'w') as f:
    f.write(f"Total training time: {elapsed_time:.2f} seconds\n")
loss_data = pd.DataFrame({'Epoch': range(1, len(train_losses) + 1), 'Training Loss': train_losses, 'Validation Loss': val_losses})
loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/{type(model).__name__}_loss_data_{timestamp}.csv', index=False)

# テスト結果の保存
test_loss_data = pd.DataFrame({'Test Loss': [test_loss], 'RMSE': [rmse], 'MAE': [mae], 'R2 Score': [r2]})
test_loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/{type(model).__name__}_test_loss_{test_loss:.4f}_{timestamp}.csv', index=False)

# データの保存
np.save(f'/home/nishioka/GNN/GNNdata/train_x_{type(model).__name__}_{timestamp}.npy', train_x.cpu().numpy())
np.save(f'/home/nishioka/GNN/GNNdata/train_y_{type(model).__name__}_{timestamp}.npy', train_y.cpu().numpy())
np.save(f'/home/nishioka/GNN/GNNdata/val_x_{type(model).__name__}_{timestamp}.npy', val_x.cpu().numpy())
np.save(f'/home/nishioka/GNN/GNNdata/val_y_{type(model).__name__}_{timestamp}.npy', val_y.cpu().numpy())
np.save(f'/home/nishioka/GNN/GNNdata/test_x_{type(model).__name__}_{timestamp}.npy', test_x.cpu().numpy())
np.save(f'/home/nishioka/GNN/GNNdata/test_y_{type(model).__name__}_{timestamp}.npy', test_y.cpu().numpy())

print("Data and labels saved successfully.")
